<a href="https://colab.research.google.com/github/korsahike/CS135-Assign1/blob/main/Spatial_engineering_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install geoalchemy2
# 1. Install system spatial dependencies for SQLite SpatiaLite
!apt-get update -qq && apt-get install -y -qq libsqlite3-mod-spatialite > /dev/null

# 2. Install required Python packages
!pip install -q geoalchemy2 geopandas shapely

from sqlalchemy import create_engine,Integer,Column,String,Float,ForeignKey,DateTime,event,text
from sqlalchemy.orm import declarative_base,relationship,sessionmaker
import datetime
import geopandas as gpd
from shapely.geometry import Point,Polygon,box
from geoalchemy2 import Geometry
import sqlite3
import pandas as pd
import numpy as np
import sqlite3


base = declarative_base()

class fleetdrive(base):
  __tablename__ = "fleetdrive"

  device_id = Column(String,primary_key=True)
  model_name = Column(String,nullable=False)
  status = Column(String,default='active')

  sessions = relationship("fleetsessions", back_populates="device")

class targetzone(base):
  __tablename__ = "targetzone"

  zone_id = Column(Integer,primary_key=True)
  model_name = Column(String,nullable=True)
  spatial_weight = Column(Float,nullable=True)

  geom = Column(Geometry(geometry_type='POINT',srid=4326))

  sessions = relationship("fleetsessions",back_populates="zone")

class fleetsessions(base):
  __tablename__ = "fleetsessions"

  session_id = Column(String,primary_key=True)
  device_id = Column(String,ForeignKey("fleetdrive.device_id"),nullable=False)
  zone_id = Column(Integer,ForeignKey("targetzone.zone_id"),nullable=False)
  start_time = Column(DateTime,default=datetime.datetime.utcnow)
  end_time = Column(DateTime,nullable=True)
  control_penalty = Column(Float,nullable=False)
  spatial_penalty = Column(Float,nullable=False)
  calibration_health_score = Column(Float,nullable=False)
  final_wsme = Column(Float,nullable=True)
  final_moran_i = Column(Float,nullable=False)
  curr_loc = Column(Geometry(geometry_type="POLYGON",srid=4326))
  device = relationship("fleetdrive",back_populates="sessions")
  zone = relationship("targetzone",back_populates="sessions")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 2.9 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:

class RobotDistance:
  def __init__(self,x_target,x_current,y_target,y_current,theta_robot,kp_linear,kp_angular,telemetry_data):
    self.x_target = np.array(x_target)
    self.y_target = np.array(y_target)
    self.x_current = np.array(x_current)
    self.y_current = np.array(y_current)
    self.theta_robot = np.array(theta_robot)
    self.kp_linear = kp_linear
    self.kp_angular = kp_angular
    self.telemetry_data = telemetry_data

  def spatial_distance(self):
    return np.sqrt((self.x_target - self.x_current)**2 + (self.y_target - self.y_current)**2)

  def target_angle(self):
    return np.arctan2(self.y_target - self.y_current,self.x_target - self.x_current)

  def angle_error_control(self):
    error = self.target_angle() - self.theta_robot
    return np.arctan2(np.sin(error),np.cos(error))

  def velocity_commands(self):
    self.v = self.kp_linear * self.spatial_distance()
    self.omega = self.kp_angular * self.angle_error_control()
    return self.v,self.omega

In [ ]:
class SpatialAutocorrelation:
  def __init__(self,N,W,xi):
    self.N = N
    self.W = np.array(W,dtype=float)
    self.xi =  np.array(xi,dtype=float)
    self.sum_xi = np.sum(self.xi)
    self.x_bar = self.sum_xi / self.N
    self.z = self.xi - self.x_bar
    self.var = self.z**2
    self.sum_var = np.sum(self.var)
    self.So = np.sum(self.W)
    self.S1 = 0.5 * np.sum((self.W +  self.W.T)**2)
    self.S2 = np.sum((np.sum(self.W,axis=1)+np.sum(self.W,axis=0))**2)
    self.lag = np.dot(self.W,self.z)
    self.cp = np.dot(self.z,self.lag)

  def morans_i(self):
    if self.So == 0 or self.sum_var == 0:
      return 0.0
    return (self.N/self.So) * (self.cp/self.sum_var)

  def morans_expected_value(self):
    return -1 /(self.N-1)

  def morans_variance(self):
    return ((self.N**2* self.S1 - self.N * self.S2 + 3 * self.So**2)/(((self.N + 1) * (self.N - 1)) * self.So**2)) - self.morans_expected_value()**2


  def morans_zscore(self):
    return (self.morans_i() - self.morans_expected_value()) / np.sqrt(self.morans_variance())

  def wmse(self,y_pred,y_init):
    y_pred = np.array(y_pred,dtype=float)
    y_init = np.array(y_init,dtype=float)
    error = y_init - y_pred
    squared_error = error**2
    return np.sum(self.W * squared_error)/np.sum(self.W)



In [ ]:
if __name__ == "__main__":

  device_id = ["Robot1","Robot2","Robot3"]
  x_target = [15,44,80]
  x_current =[12.5,45.1,80.3]
  y_target = [25,52,90]
  y_current = [22.4,55.6,88.1]
  theta_robot = [0.1, -0.5, 0.75]
  kp_linear,kp_angular = 0.5,0.8

  telemetry = pd.DataFrame({
      "device_id":device_id,
      "x_target":x_target,
      "x_current":x_current,
      "y_target": y_target,
      "y_current": y_current,
      "theta_robot":theta_robot})

  robot_features = RobotDistance(x_target,x_current,y_target,y_current,theta_robot,kp_linear,kp_angular,telemetry)
  v_metric,omega = robot_features.velocity_commands()
  sp = robot_features.spatial_distance()
  telemetry["v_metric"] = v_metric
  telemetry["omega"] = omega
  telemetry["Spatial_distance"] = sp
  curr_coords = [Point(-73.3899,40.7338),Point(-74.2234,40.8932),Point(-74.4428,40.7845)]
  tar_coords = [Point(-73.3967,40.7788),Point(-74.4335,40.9112),Point(-74.4933,40.7546)]
  display(telemetry)

  W = [
    [0.0, 0.5, 0.5],
    [0.4, 0.0, 0.6],
    [0.2, 0.8, 0.0]
]
N = len(W)
xi = [1.2, 8.5, 1.5]

spatial = SpatialAutocorrelation(N=N, W=W, xi=xi)
# 1. Ensure the container has the spatial library installed
!apt-get update -qq && apt-get install -y -qq libsqlite3-mod-spatialite > /dev/null



# 2. Create the engine cleanly
engine = create_engine("sqlite:///:memory:", echo=False)

# 3. Create the listener to load the extension module
@event.listens_for(engine, "connect")
def load_spatialite(dbapi_connection, connection_record):
    if isinstance(dbapi_connection, sqlite3.Connection):
        dbapi_connection.enable_load_extension(True)
        dbapi_connection.load_extension('/usr/lib/x86_64-linux-gnu/mod_spatialite.so')

# 4. Correctly initialize the metadata inside the engine's own connection context
with engine.connect() as conn:
    # Use text() to safely execute the spatial configuration query
    conn.execute(text("SELECT InitSpatialMetaData(1);"))
    conn.commit()

# 5. Now create the database tables safely
base.metadata.create_all(engine)

base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
db_session = Session()

print("Morans I:", spatial.morans_i())
print("Morans E(X):", spatial.morans_expected_value())
print("Morans zscore:", spatial.morans_zscore())

for i, device_identifier in enumerate(device_id):
  db_session.merge(fleetdrive(device_id= device_identifier,model_name="RobotX"))
  zone_inputs = targetzone(zone_id=i+1,model_name=device_identifier,spatial_weight=float(W[i][i]),geom=f"SRID=4326;{tar_coords[i].wkt}")
  db_session.merge(zone_inputs)
  poly_geom = box(curr_coords[i].x - 0.01, curr_coords[i].y - 0.01, curr_coords[i].x + 0.01, curr_coords[i].y + 0.01)

  session_record = fleetsessions(

      session_id=f"sess_{device_identifier}_{i}",
      device_id=device_identifier,
      zone_id=i+1,
      control_penalty=float(telemetry.loc[i, "omega"]),
      spatial_penalty=float(telemetry.loc[i, "Spatial_distance"]),
      calibration_health_score=max(0.0, 100.0 - float(telemetry.loc[i, "Spatial_distance"])),
      final_wsme=spatial.wmse(y_pred=x_current, y_init=x_target),
      final_moran_i=spatial.morans_i(),
      curr_loc=f"SRID=4326;{poly_geom.wkt}"

  )
  db_session.merge(session_record)
db_session.commit()
print("\nSuccessfully ran processing metrics and saved sessions into the DB.")

,device_id,x_target,x_current,y_target,y_current,theta_robot,v_metric,omega,Spatial_distance
0,Robot1,15,12.5,25,22.4,0.10,1.803469,0.564003,3.606938
1,Robot2,44,45.1,52,55.6,-0.50,1.882153,-1.093874,3.764306
2,Robot3,80,80.3,90,88.1,0.75,0.961769,0.781919,1.923538


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Morans I: -0.639128736081266
Morans E(X): -0.5
Morans zscore: -0.9451936359262935

Successfully ran processing metrics and saved sessions into the DB.


In [ ]:

import folium

m = folium.Map(location=[40.8, -74.0], zoom_start=10, tiles="OpenStreetMap")

for i in range(len(device_id)):
    folium.Marker(
        location=[tar_coords[i].y, tar_coords[i].x],
        popup=f"Target Zone {i+1} ({device_id[i]})",
        icon=folium.Icon(color="green", icon="flag")
    ).add_to(m)

    poly_geom = box(curr_coords[i].x - 0.01, curr_coords[i].y - 0.01, curr_coords[i].x + 0.01, curr_coords[i].y + 0.01)
    polygon_coordinates = [[y, x] for x, y in poly_geom.exterior.coords]

    folium.Polygon(
        locations=polygon_coordinates,
        color="blue",
        weight=2,
        fill=True,
        fill_opacity=0.25,
        popup=f"Current Location Area: {device_id[i]}"
    ).add_to(m)

m
